# Stage 1: Domain-Adaptive Pretraining (DAPT) for GraphCodeBERT

This notebook is optimized for a local workstation (Ryzen 5 5600, 32GB RAM, RTX 3060 12GB) to "teach" GraphCodeBERT x86/x64 Assembly using Masked Language Modeling (MLM).
We use a dataset generator to stream the 22GB dataset so it comfortably fits in your 32GB of system RAM, and training arguments optimized for your 12GB VRAM GPU.

In [ ]:
# Ensure you have the required libraries installed in your local Python environment:
# Run this in your VS Code terminal:
# pip install transformers datasets accelerate torch

In [ ]:
import os

# Point this directly to your local JSON output folder
DATA_DIR = r'e:\Thesis\combined_output'

### 1. Data Loading and Chunking
GraphCodeBERT handles a maximum of 512 tokens. We need to extract the assembly instructions and chunk them into sequences of ~512 tokens.

In [ ]:
import glob
import json
from datasets import Dataset

def generate_assembly_chunks(data_dir, chunk_size=400, max_files=None):
    """
    Generator that reads JSON files and yields chunks of assembly instructions.
    Uses chunk_size=400 to account for GraphCodeBERT's 512 token limit.
    """
    json_files = glob.glob(os.path.join(data_dir, '*.json'))
    
    if max_files:
        json_files = json_files[:max_files]
        print(f"Limiting to the first {max_files} files for a test run...")
    
    for file_path in json_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            if 'asm' not in data:
                continue
                
            raw_lines = data['asm'].split('\n')
            instructions = []
            
            for line in raw_lines:
                clean_line = line.strip()
                # Ignore comments (;) and empty lines to keep pure assembly
                if clean_line and not clean_line.startswith(';'):
                    instructions.append(clean_line)
            
            # Break into 400-instruction chunks
            for i in range(0, len(instructions), chunk_size):
                chunk = " \n ".join(instructions[i:i + chunk_size])
                yield {"text": chunk}
                
        except Exception as e:
            pass # Suppress output so it doesn't spam for broken files

# Create Hugging Face Dataset from generator
# You can set max_files=50 initially to do a quick 3-minute test run, 
# then change it to max_files=None when you want to train on the entire 22GB.
dataset = Dataset.from_generator(lambda: generate_assembly_chunks(DATA_DIR, max_files=None))
print(f"Created dataset with {len(dataset)} assembly chunks.")

### 2. Tokenization and Data Collator
We use GraphCodeBERT's tokenizer. For MLM, we use `DataCollatorForLanguageModeling` which randomly masks 15% of the tokens.

In [ ]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

MODEL_NAME = "microsoft/graphcodebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Masking 15% of the tokens (standard for BERT/MLM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=True, 
    mlm_probability=0.15
)

### 3. Initialize Model and Train
We load `AutoModelForMaskedLM` and use Hugging Face's `Trainer` to run the finetuning loop.

In [ ]:
from transformers import AutoModelForMaskedLM, TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
import torch
import os

# Load the base model with a Masked Language Modeling head
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

# Split dataset into train and validation (90/10 split)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Explicitly use FP16 for the RTX 3060 (Ampere architecture excels at FP16 mixed precision)
# Your 12GB VRAM is ample for per_device_train_batch_size=8, but we use
# gradient_accumulation_steps=2 to mimic an effective batch size of 16 for better gradients.
# Your Ryzen 5 5600 has 12 threads. We use 4 dedicated workers for fast I/O processing.
training_args = TrainingArguments(
    output_dir="./gcb-assembly-pretrained",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2, 
    per_device_eval_batch_size=8,
    dataloader_num_workers=4, 
    eval_strategy="steps",
    eval_steps=1000,
    save_steps=5000,
    logging_steps=100,
    learning_rate=5e-5,
    fp16=True, 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# Empty PyTorch cache just in case
torch.cuda.empty_cache()

# Detect last checkpoint to allow pausing and resuming
last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)
    if last_checkpoint is not None:
        print(f"Resuming training from {last_checkpoint}")

# Start or resume training
trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
# Save the final pretrained model and tokenizer locally
trainer.save_model("./gcb-assembly-final")
tokenizer.save_pretrained("./gcb-assembly-final")
print("Training Complete! Valid x86 GCB Model saved in ./gcb-assembly-final")